In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
# InceptionResNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


In [41]:
# 데이터 로드
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [42]:
train.shape

(769, 1026)

In [43]:
# Feature(X)와 Target(y) 분리
X = train.iloc[:, 2:].values.reshape(-1, 32, 32, 1)  # 1채널 이미지 데이터
y = train["label"].values  # 분류할 대상 라벨

In [44]:
# 라벨을 숫자로 변환 (Label Encoding)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

In [45]:
# 데이터 분할
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.1, random_state=42)

In [46]:
# 1채널 이미지를 3채널로 변환
X_train_rgb = np.concatenate([X_train, X_train, X_train], axis=-1)  # 3채널로 복제
X_valid_rgb = np.concatenate([X_valid, X_valid, X_valid], axis=-1)  # 3채널로 복제

# 테스트 데이터 처리
X_test = test.iloc[:, 1:].values.reshape(-1, 32, 32, 1)  # 1채널 이미지 데이터
X_test_rgb = np.concatenate([X_test, X_test, X_test], axis=-1)  # 3채널로 변환

# 데이터 정규화
X_train_rgb = X_train_rgb / 255.0  # 정규화: 픽셀 값을 0과 1 사이로 변환
X_valid_rgb = X_valid_rgb / 255.0  # 정규화
X_test_rgb = X_test_rgb / 255.0  # 정규화

# # 데이터 증강 설정
# datagen = ImageDataGenerator(
#     rotation_range=20,        # 이미지 회전
#     width_shift_range=0.2,    # 수평 이동
#     height_shift_range=0.2,   # 수직 이동
#     shear_range=0.2,          # 전단 변환
#     zoom_range=0.2,           # 확대/축소
#     horizontal_flip=True,      # 수평 뒤집기
#     fill_mode='nearest'       # 빈 공간 채우기
# )

# # 데이터 증강을 적용하여 학습 데이터 생성기 만들기
# train_generator = datagen.flow(X_train_rgb, y_train, batch_size=32)

In [49]:

base_model = ResNet50  (weights=None, include_top=False, input_shape=(32, 32, 3))  # 3채널 입력

# 모델 구성
model = models.Sequential([
    layers.InputLayer(input_shape=(32, 32, 3)),  # 3채널로 입력
    base_model,
    layers.GlobalMaxPool2D (),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(len(label_encoder.classes_), activation='softmax')  # 클래스 수에 맞춰 출력층
])

model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)                │ (None, 1, 1, 2048)          │      23,587,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling2d_4               │ (None, 2048)                │               0 │
│ (GlobalMaxPooling2D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 2048)                │           8,192 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 256)                 │         524,544 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ (None, 10)                  │           2,570 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 24,123,018 (92.02 MB)

 Trainable params: 24,065,802 (91.80 MB)

 Non-trainable params: 57,216 (223.50 KB)

In [50]:
# 모델 컴파일 (learning rate 설정)
learning_rate = 0.0005
optimizer = Adam(learning_rate=learning_rate)
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 콜백 설정
early_stopping = EarlyStopping(monitor='val_loss', patience=200, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('best_mobilenet_model.keras', monitor='val_loss', save_best_only=True)

# 모델 학습
model.fit(
    X_train_rgb, y_train,
    validation_data=(X_valid_rgb, y_valid),
    epochs=500,
    callbacks=[early_stopping, model_checkpoint]
)


Epoch 1/500
22/22 ━━━━━━━━━━━━━━━━━━━━ 111s 2s/step - accuracy: 0.1372 - loss: 3.2846 - val_accuracy: 0.1039 - val_loss: 2.2366
Epoch 2/500
22/22 ━━━━━━━━━━━━━━━━━━━━ 20s 36ms/step - accuracy: 0.2820 - loss: 2.7402 - val_accuracy: 0.1299 - val_loss: 2.2412
Epoch 3/500
22/22 ━━━━━━━━━━━━━━━━━━━━ 11s 480ms/step - accuracy: 0.4444 - loss: 1.9256 - val_accuracy: 0.2208 - val_loss: 2.2020
Epoch 4/500
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.5940 - loss: 1.4012 - val_accuracy: 0.1688 - val_loss: 2.3484
Epoch 5/500
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.6622 - loss: 1.1776 - val_accuracy: 0.1818 - val_loss: 2.3549
Epoch 6/500
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.7911 - loss: 0.8171 - val_accuracy: 0.1818 - val_loss: 2.2464
Epoch 7/500
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.8098 - loss: 0.6572 - val_accuracy: 0.1688 - val_loss: 2.3642
Epoch 8/500
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.8900 - loss: 0.3874 - val_accuracy:

In [51]:
# 최적 모델 로드
model.load_weights('best_mobilenet_model.keras')  # 수정된 파일 이름

# 모델 평가
loss, accuracy = model.evaluate(X_valid_rgb, y_valid)
print(f"Validation accuracy: {accuracy:.4f}")




3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9441 - loss: 0.1463
Validation accuracy: 0.9351


In [52]:
# 테스트 데이터에 대한 예측
y_test_pred = model.predict(X_test_rgb)
y_test_pred_labels = np.argmax(y_test_pred, axis=1)  # 예측된 라벨

# 예측된 라벨을 원래 라벨로 변환
y_test_pred_labels = label_encoder.inverse_transform(y_test_pred_labels)

# 제출 파일 생성
submission = pd.read_csv('sample_submission.csv')
submission['label'] = y_test_pred_labels  # 예측된 라벨로 업데이트


8/8 ━━━━━━━━━━━━━━━━━━━━ 9s 630ms/step


In [53]:
submission.to_csv('./Resnet_0308_5.csv', index=False, encoding='utf-8-sig')

In [54]:
submission

,ID,label
0,TEST_000,airplane
1,TEST_001,cat
2,TEST_002,emotion_face
3,TEST_003,building
4,TEST_004,truck
...,...,...
245,TEST_245,police_car
246,TEST_246,cat
247,TEST_247,building
248,TEST_248,police_car
